# OCR model comparison for VL-1.6-style tasks — small vs medium vs StructureV3

One notebook, one Kaggle T4. Goal: **find the lightest PaddleOCR model that best reproduces PaddleOCR-VL 1.6** on *our* task (extract all text + its format) over the **test-dataset-100** images.

We run four models one after another on the same 100 images:

| Model | Role | Produces |
|---|---|---|
| PP-OCRv6 small | candidate | flat text lines |
| PP-OCRv6 medium | candidate | flat text lines |
| PP-StructureV3 | candidate | Markdown + **HTML** (tables + layout) |
| **PaddleOCR-VL 1.6** | **reference** | Markdown + **HTML** (tables + layout) |

There is **no ground-truth text** in this dataset, so we use **VL 1.6 as a pseudo-reference** and also report GT-free proxies:
text closeness to VL (word P/R/F1 + char similarity), format capture (tables vs VL), recognition confidence, cross-model agreement, and speed / GPU / disk.

Only **5 sample images** are shown inline (one per category); everything else is charts + a downloadable **ZIP**.

**Kaggle setup:** Accelerator = `GPU T4 x2` · Internet = On · attach the `test-dataset-100` dataset · then **Run All** (use *Restart & Run All* once if a paddle import errors after install).

In [1]:
# ============================ QUIET + CONFIG ============================
# Silence PaddlePaddle/PaddleX/paddleformers noise so the run is warning-free.
# These env vars MUST be set before `import paddle` (first done in the VERIFY cell).
import os, warnings, logging
os.environ["GLOG_minloglevel"] = "2"                       # hide glog INFO/WARNING (e.g. GPU-cap note)
os.environ["GLOG_v"] = "0"
os.environ["FLAGS_call_stack_level"] = "0"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"  # skip "Checking connectivity..." message
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)                           # drop INFO/WARNING from all std loggers
for _n in ("paddle", "paddlex", "paddleocr", "ppocr", "paddleformers", "paddlenlp"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# ============================ CONFIG ============================
# Images come from an attached Kaggle Dataset. The loader searches INPUT_DIR
# RECURSIVELY, so nested subfolders are fine. manifest.csv (categories) is read if present.
INPUT_DIR = "/kaggle/input/datasets/amasikifthakerhamim/test-dataset-100"

OUTPUT_ROOT = "/kaggle/working/ocr_compare"      # all model outputs + metrics + figures live here
ZIP_PATH    = "/kaggle/working/ocr_compare_results"   # .zip appended automatically

N_SAMPLE_SHOW = 5      # how many sample images to display inline (one per category)

# ---- Reference model (PaddleOCR-VL 1.6) ----
# RUN_VL = True  -> run VL 1.6 in THIS notebook to produce the reference (heavy: ~15 GB, ~18 s/img).
# RUN_VL = False -> skip running VL and read precomputed VL Markdown from VL_REF_DIR instead.
RUN_VL     = True
VL_REF_DIR = ""        # only used when RUN_VL = False: folder holding VL's *.md (per image)

# ---- PP-StructureV3 config (fidelity-first: HTML that replicates text + layout) ----
# Task = parse the image into HTML with faithful text AND layout/tables.
# Decision: KEEP the strong default OCR (PP-OCRv5_server) — best text fidelity. The text
# StructureV3 lost in the first run was dropped by the LAYOUT stage, not the OCR model, so
# we widen layout/detection coverage instead of downgrading OCR. Unused modules
# (formula/chart/seal) are OFF — this dataset has none, and they cost ~3 GB + most of the load.
STRUCT_KW = dict(
    use_table_recognition=True,        # HTML tables — core to layout replication
    use_region_detection=True,         # reading order for faithful layout
    use_formula_recognition=False,     # none here -> drops PP-FormulaNet_plus-L
    use_chart_recognition=False,       # none here -> drops the PP-Chart2Table VLM
    use_seal_recognition=False,        # none here -> drops seal det/rec
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    text_det_limit_type="max",
    text_det_limit_side_len=1216,      # higher res -> detects smaller text (default ~960)
    layout_threshold=0.3,              # more inclusive layout regions (default ~0.5)
    device="gpu:0",
)
# Flip to True to ALSO run a StructureV3 variant with PP-OCRv6_medium OCR (A/B the OCR swap
# you asked about). Costs one extra model run.
STRUCT_OCRV6_AB = False

# The candidate models (VL is handled separately as the reference).
MODELS = [
    {"key": "ocrv6_small",  "kind": "ocr",    "name": "PP-OCRv6_small",
     "det": "PP-OCRv6_small_det",  "rec": "PP-OCRv6_small_rec",  "params": "7.7M"},
    {"key": "ocrv6_medium", "kind": "ocr",    "name": "PP-OCRv6_medium",
     "det": "PP-OCRv6_medium_det", "rec": "PP-OCRv6_medium_rec", "params": "34.5M"},
    {"key": "structurev3",  "kind": "struct", "name": "PP-StructureV3 (OCRv5-server)",
     "struct_kwargs": STRUCT_KW},
]
if STRUCT_OCRV6_AB:
    MODELS.append({"key": "structurev3_ocrv6", "kind": "struct", "name": "PP-StructureV3 (OCRv6-medium)",
                   "struct_kwargs": dict(STRUCT_KW, text_detection_model_name="PP-OCRv6_medium_det",
                                         text_recognition_model_name="PP-OCRv6_medium_rec")})

import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("Output root:", OUTPUT_ROOT, "| StructureV3 OCRv6 A/B:", STRUCT_OCRV6_AB)

Output root: /kaggle/working/ocr_compare | StructureV3 OCRv6 A/B: False


In [2]:
# ============================ INSTALL ============================
# PaddleOCR (doc-parser extras) + GPU PaddlePaddle 3.2.1 (CUDA 12.6 wheel); runs on the T4.
# We uninstall torch afterwards: paddlepaddle-gpu (cu126) downgrades nvidia-*-cu12 to 12.6 which
# breaks Kaggle's preinstalled torch ("undefined symbol: ncclCommShrink"), and `import paddleocr`
# pulls torch via modelscope. We don't use torch here, so removing it lets paddleocr import cleanly.
!pip install -q "paddleocr[doc-parser]"
!pip install -q "paddlepaddle-gpu==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ --timeout 180 --retries 10
!pip install -q nvidia-ml-py markdown
!pip -q uninstall -y torch torchvision torchaudio
print("\nInstall finished. torch removed on purpose (unused here); earlier pip 'conflict' lines are resolved by this.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 26.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 88.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 82.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [3]:
# ============================ VERIFY GPU ============================
import warnings, logging
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import paddle, importlib
print("paddlepaddle :", paddle.__version__)
print("CUDA compiled:", paddle.device.is_compiled_with_cuda())
print("GPU count    :", paddle.device.cuda.device_count())
assert paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() >= 1, (
    "PaddlePaddle does not see a GPU. Set Accelerator = 'GPU T4 x2', enable Internet, then Restart & Run All.")
_x = paddle.to_tensor([1.0, 2.0, 3.0], place=paddle.CUDAPlace(0))
print("GPU tensor sum:", float((_x * 2).sum()), "-> PaddlePaddle GPU is working.")
assert importlib.util.find_spec("torch") is None, (
    "torch is still installed and will break the paddleocr import via modelscope. Re-run INSTALL, then Restart & Run All.")
import io, contextlib
with contextlib.redirect_stderr(io.StringIO()):   # swallow one-time "PyTorch was not found" notice
    import paddleocr
print("paddleocr    :", paddleocr.__version__, "-> import OK.")

paddlepaddle : 3.2.1
CUDA compiled: True
GPU count    : 2
GPU tensor sum: 12.0 -> PaddlePaddle GPU is working.
paddleocr    : 3.7.0 -> import OK.


In [4]:
# ============================ GET IMAGES + CATEGORIES ============================
import os, glob, csv
EXTS = ("jpg", "jpeg", "png", "bmp", "webp", "jfif", "tif", "tiff")

def find_local_images(root):
    got = []
    for e in EXTS:
        got += glob.glob(os.path.join(root, "**", f"*.{e}"), recursive=True)
        got += glob.glob(os.path.join(root, "**", f"*.{e.upper()}"), recursive=True)
    return sorted(set(got))

images = find_local_images(INPUT_DIR)
assert images, (f"No images found under {INPUT_DIR!r}. Attach the test-dataset-100 dataset (Add Input) "
                "and check the path in CONFIG.")

# category map from manifest.csv (filename,category,...) if present; else derive from filename prefix
CATEGORY = {}
for mf in glob.glob(os.path.join(INPUT_DIR, "**", "manifest.csv"), recursive=True):
    with open(mf, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("filename"):
                CATEGORY[row["filename"]] = row.get("category", "other")
def cat_of(path):
    b = os.path.basename(path)
    if b in CATEGORY: return CATEGORY[b]
    return b.rsplit("_", 1)[0] if "_" in b else "other"

from collections import Counter
cats = Counter(cat_of(p) for p in images)
print(f"{len(images)} images found under {INPUT_DIR}")
print("By category:", dict(sorted(cats.items())))

# pick sample images to display inline later: one per category, up to N_SAMPLE_SHOW
SAMPLES, seen = [], set()
for p in images:
    c = cat_of(p)
    if c not in seen:
        seen.add(c); SAMPLES.append(p)
    if len(SAMPLES) >= N_SAMPLE_SHOW: break
print("Inline samples:", [os.path.basename(p) for p in SAMPLES])

100 images found under /kaggle/input/datasets/amasikifthakerhamim/test-dataset-100
By category: {'agenda': 23, 'form': 23, 'invoice': 23, 'original': 11, 'report': 20}
Inline samples: ['agenda_01.jpg', 'form_01.jpg', 'invoice_01.jpg', 'original_001.jpg', 'report_01.jpg']


In [5]:
# ============================ BENCH UTILS + METRIC HELPERS ============================
import os, sys, time, threading, subprocess, gc, re, html, difflib, contextlib, warnings, logging
from collections import Counter
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)

LOG_DIR = os.path.join(OUTPUT_ROOT, "logs"); os.makedirs(LOG_DIR, exist_ok=True)

@contextlib.contextmanager
def quiet(logfile):
    """Redirect C-level stderr (fd 2) + Python stderr into a logfile during noisy model
    load/predict, so PaddlePaddle/PaddleX chatter never reaches the cell. Real Python
    exceptions still propagate (they don't go through this stream), so errors stay visible."""
    f = open(logfile, "a"); old_fd = None; old_py = sys.stderr
    try:
        sys.stderr.flush(); old_fd = os.dup(2); os.dup2(f.fileno(), 2); sys.stderr = f
        yield
    finally:
        try: sys.stderr.flush()
        except Exception: pass
        sys.stderr = old_py
        if old_fd is not None:
            try: os.dup2(old_fd, 2); os.close(old_fd)
            except Exception: pass
        f.close()

try:
    import pynvml; pynvml.nvmlInit(); _H = pynvml.nvmlDeviceGetHandleByIndex(0); _NVML = True
except Exception as e:
    _NVML = False; print("pynvml unavailable, GPU memory will read NaN:", e)

def gpu_used_mb():
    return pynvml.nvmlDeviceGetMemoryInfo(_H).used / 1024**2 if _NVML else float("nan")

class GPUSampler:
    """Context manager: polls GPU memory and records peak used (MB)."""
    def __init__(self, interval=0.05):
        self.interval, self.peak, self._run, self._t = interval, 0.0, False, None
    def _loop(self):
        while self._run:
            try: self.peak = max(self.peak, gpu_used_mb())
            except Exception: pass
            time.sleep(self.interval)
    def __enter__(self):
        self.peak = gpu_used_mb() if _NVML else 0.0
        self._run = True; self._t = threading.Thread(target=self._loop, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._run = False
        if self._t: self._t.join()

def dir_size_mb(path):
    path = os.path.expanduser(path); total = 0
    for root, _, files in os.walk(path):
        for f in files:
            try: total += os.path.getsize(os.path.join(root, f))
            except OSError: pass
    return total / 1024**2

MODEL_CACHE_DIRS = ["~/.paddlex", "~/.paddleocr", "~/.cache/huggingface", "~/.cache/paddle"]
def model_cache_mb():
    return sum(dir_size_mb(d) for d in MODEL_CACHE_DIRS)

def free_gpu(predictor=None):
    """Release a model and return GPU memory so the next (heavier) model fits on the T4."""
    try:
        if predictor is not None: del predictor
    except Exception: pass
    gc.collect()
    try: paddle.device.cuda.empty_cache()
    except Exception: pass
    time.sleep(1.0)

def show_env():
    q = "name,memory.total,driver_version,compute_cap"
    out = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    print("GPU:", out)
show_env()

# ---------- text extraction + normalization (for quality-vs-VL metrics) ----------
def _deep_find(obj, key):
    """First value for `key` anywhere in a nested dict/list, else None."""
    if isinstance(obj, dict):
        if key in obj: return obj[key]
        for v in obj.values():
            r = _deep_find(v, key)
            if r is not None: return r
    elif isinstance(obj, list):
        for v in obj:
            r = _deep_find(v, key)
            if r is not None: return r
    return None

def extract_from_res(res):
    """Return dict: rec_texts, rec_scores, md (markdown text), n_tables."""
    try: j = res.json
    except Exception: j = {}
    if isinstance(j, dict) and set(j.keys()) == {"res"}: j = j["res"]
    rec_texts  = _deep_find(j, "rec_texts")  or []
    rec_scores = _deep_find(j, "rec_scores") or []
    md = ""
    m = getattr(res, "markdown", None)
    if isinstance(m, dict): md = m.get("markdown_texts", "") or ""
    elif isinstance(m, str): md = m
    if not md:  # VL fallback: stitch reading-ordered block contents
        blocks = _deep_find(j, "parsing_res_list") or []
        md = "\n".join(b.get("block_content", "") for b in blocks if isinstance(b, dict))
    n_tables = md.lower().count("<table")
    return {"rec_texts": rec_texts, "rec_scores": rec_scores, "md": md, "n_tables": n_tables}

_TAG = re.compile(r"<[^>]+>")
_MD  = re.compile(r"[#*_`|>]+")
_WS  = re.compile(r"\s+")
def normtext(s):
    """Strip HTML/Markdown to a lowercase plain-text token stream for fair text comparison."""
    s = _TAG.sub(" ", s or ""); s = html.unescape(s); s = _MD.sub(" ", s)
    s = s.replace("-", " "); s = _WS.sub(" ", s).lower().strip()
    return s

def char_sim(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def word_prf(pred, ref):
    """Word-level precision/recall/F1 of `pred` against reference `ref` (multiset overlap)."""
    p, r = Counter(pred.split()), Counter(ref.split())
    inter = sum((p & r).values())
    P = inter / max(sum(p.values()), 1)
    R = inter / max(sum(r.values()), 1)
    F = 2 * P * R / max(P + R, 1e-9)
    return P, R, F

# ---------- Markdown -> full HTML page (deliverable: HTML that replicates text + layout) ----------
try:
    import markdown as _mdlib; _HAS_MD = True
except Exception:
    _HAS_MD = False
_HTML_CSS = ("body{font-family:Arial,Helvetica,sans-serif;max-width:900px;margin:24px auto;"
             "padding:0 16px;line-height:1.5}table{border-collapse:collapse;margin:12px 0}"
             "td,th{border:1px solid #888;padding:4px 8px;vertical-align:top}"
             "h1,h2,h3{margin:.6em 0 .3em}img{max-width:100%}")
def md_to_html(md_text, title):
    """StructureV3/VL Markdown (with embedded <table> HTML) -> a standalone HTML page."""
    if _HAS_MD:
        body = _mdlib.markdown(md_text or "", extensions=["tables", "md_in_html", "sane_lists"])
    else:
        body = "<pre>" + (md_text or "") + "</pre>"
    return (f"<!doctype html>\n<html><head><meta charset='utf-8'><title>{title}</title>"
            f"<style>{_HTML_CSS}</style></head>\n<body>\n{body}\n</body></html>\n")
print("helpers ready.")

GPU: Tesla T4, 15360 MiB, 580.159.04, 7.5
Tesla T4, 15360 MiB, 580.159.04, 7.5
helpers ready.


## Run the candidate models
Each model loads, runs on all 100 images (outputs saved to disk), then its GPU memory is freed before the next.
StructureV3 keeps the strong default OCR (PP-OCRv5_server) with formula/chart/seal modules OFF and wider layout/detection coverage, and writes an HTML page per image. Per-image text, confidence, region counts and latency are collected for the metrics below.

In [6]:
# ============================ LOAD + RUN CANDIDATES ============================
import json, os, time

def build_predictor(m):
    if m["kind"] == "ocr":
        from paddleocr import PaddleOCR
        return PaddleOCR(text_detection_model_name=m["det"], text_recognition_model_name=m["rec"],
                         use_doc_orientation_classify=False, use_doc_unwarping=False,
                         use_textline_orientation=False, device="gpu:0"), ["save_to_json", "save_to_img"]
    if m["kind"] == "struct":
        from paddleocr import PPStructureV3
        return PPStructureV3(**m["struct_kwargs"]), ["save_to_json", "save_to_img", "save_to_markdown"]
    raise ValueError(m["kind"])

RESULTS  = {}   # key -> {image: {latency, regions, chars, mean_conf, norm_text, md, n_tables, category}}
RESOURCE = {}   # key -> resource/summary dict

def run_model(m):
    key, name = m["key"], m["name"]
    out_dir = os.path.join(OUTPUT_ROOT, key); os.makedirs(out_dir, exist_ok=True)
    print(f"\n=== {name} ===")
    logf = os.path.join(LOG_DIR, key + ".log")
    cache_before = model_cache_mb(); t0 = time.time()
    with GPUSampler() as g_load, quiet(logf):
        predictor, save_fns = build_predictor(m)
    load_time = time.time() - t0
    model_disk = model_cache_mb() - cache_before
    gpu_after_load = gpu_used_mb()
    print(f"  loaded in {load_time:.1f}s | model disk {model_disk:.0f} MB | GPU after load {gpu_after_load:.0f} MB")

    per_image = {}; N = len(images)
    with GPUSampler() as g_run, quiet(logf):
        for i, p in enumerate(images, 1):
            b = os.path.basename(p); t = time.time()
            results = list(predictor.predict(str(p)))
            dt = time.time() - t
            texts, scores, md_all, ntab = [], [], "", 0
            for res in results:
                for fn in save_fns:
                    try: getattr(res, fn)(save_path=out_dir)
                    except Exception as e: print(f"    {fn} failed on {b}: {e}")
                ex = extract_from_res(res)
                texts += ex["rec_texts"]; scores += ex["rec_scores"]
                md_all += ("\n" + ex["md"] if ex["md"] else ""); ntab += ex["n_tables"]
            if m["kind"] == "struct" and md_all.strip():   # deliverable: HTML replicating layout
                with open(os.path.join(out_dir, os.path.splitext(b)[0] + ".html"), "w", encoding="utf-8") as _hf:
                    _hf.write(md_to_html(md_all, b))
            plain = md_all if md_all.strip() else " ".join(texts)
            per_image[b] = {"latency": round(dt, 3), "regions": len(texts),
                            "chars": len(normtext(plain)),
                            "mean_conf": round(sum(scores)/len(scores), 4) if scores else float("nan"),
                            "norm_text": normtext(plain), "n_tables": ntab, "category": cat_of(p)}
            print(f"\r  [{i:>3}/{N}] {b:<28}", end="")
    print()
    peak_run = g_run.peak
    lat = [v["latency"] for v in per_image.values()]
    RESULTS[key] = per_image
    RESOURCE[key] = {"model": name, "num_images": N,
                     "avg_latency_s": round(sum(lat)/len(lat), 3), "total_infer_s": round(sum(lat), 1),
                     "gpu_peak_MB": round(peak_run, 1), "gpu_after_load_MB": round(gpu_after_load, 1),
                     "model_disk_MB": round(model_disk, 1), "output_disk_MB": round(dir_size_mb(out_dir), 1),
                     "load_time_s": round(load_time, 1)}
    print(f"  avg latency {RESOURCE[key]['avg_latency_s']:.2f}s | peak GPU {peak_run:.0f} MB")
    free_gpu(predictor)
    print(f"  GPU after free: {gpu_used_mb():.0f} MB")

for m in MODELS:
    run_model(m)


=== PP-OCRv6_small ===
  loaded in 31.6s | model disk 30 MB | GPU after load 595 MB
  [100/100] report_20.jpg               
  avg latency 0.46s | peak GPU 2139 MB
  GPU after free: 911 MB

=== PP-OCRv6_medium ===
  loaded in 33.4s | model disk 133 MB | GPU after load 987 MB
  [100/100] report_20.jpg               
  avg latency 0.78s | peak GPU 3761 MB
  GPU after free: 1317 MB

=== PP-StructureV3 (OCRv5-server) ===
  loaded in 297.3s | model disk 1024 MB | GPU after load 2109 MB
  [100/100] report_20.jpg               
  avg latency 2.03s | peak GPU 4485 MB
  GPU after free: 3303 MB


## Run the reference (PaddleOCR-VL 1.6)
VL 1.6 is the pseudo-ground-truth. It runs last on a freed GPU (it needs ~15 GB). Set `RUN_VL = False` in CONFIG to instead read precomputed VL Markdown from `VL_REF_DIR`.

In [7]:
# ============================ REFERENCE: PaddleOCR-VL 1.6 ============================
import os, glob, time, json
VL_KEY = "vl_1_6"
vl_dir = os.path.join(OUTPUT_ROOT, VL_KEY); os.makedirs(vl_dir, exist_ok=True)
VL_TEXT = {}   # image basename -> normalized VL text
VL_META = {}   # image basename -> {n_tables}

if RUN_VL:
    print("=== PaddleOCR-VL 1.6 (reference) ===")
    logf = os.path.join(LOG_DIR, VL_KEY + ".log")
    cache_before = model_cache_mb(); t0 = time.time()
    with GPUSampler() as g_load, quiet(logf):
        from paddleocr import PaddleOCRVL
        predictor = PaddleOCRVL(pipeline_version="v1.6", device="gpu:0")
    load_time = time.time() - t0; model_disk = model_cache_mb() - cache_before
    print(f"  loaded in {load_time:.1f}s | model disk {model_disk:.0f} MB | GPU after load {gpu_used_mb():.0f} MB")
    per_image = {}; N = len(images)
    with GPUSampler() as g_run, quiet(logf):
        for i, p in enumerate(images, 1):
            b = os.path.basename(p); t = time.time()
            results = list(predictor.predict(str(p))); dt = time.time() - t
            md_all, ntab = "", 0
            for res in results:
                for fn in ["save_to_json", "save_to_img", "save_to_markdown"]:
                    try: getattr(res, fn)(save_path=vl_dir)
                    except Exception as e: print(f"    {fn} failed on {b}: {e}")
                ex = extract_from_res(res); md_all += ("\n" + ex["md"] if ex["md"] else ""); ntab += ex["n_tables"]
            if md_all.strip():   # VL reference HTML, for HTML-to-HTML comparison
                with open(os.path.join(vl_dir, os.path.splitext(b)[0] + ".html"), "w", encoding="utf-8") as _hf:
                    _hf.write(md_to_html(md_all, b))
            VL_TEXT[b] = normtext(md_all); VL_META[b] = {"n_tables": ntab}
            per_image[b] = {"latency": round(dt, 3), "n_tables": ntab, "category": cat_of(p)}
            print(f"\r  [{i:>3}/{N}] {b:<28}", end="")
    print()
    lat = [v["latency"] for v in per_image.values()]
    RESOURCE[VL_KEY] = {"model": "PaddleOCR-VL-1.6", "num_images": N,
                        "avg_latency_s": round(sum(lat)/len(lat), 3), "total_infer_s": round(sum(lat), 1),
                        "gpu_peak_MB": round(g_run.peak, 1), "gpu_after_load_MB": float("nan"),
                        "model_disk_MB": round(model_disk, 1), "output_disk_MB": round(dir_size_mb(vl_dir), 1),
                        "load_time_s": round(load_time, 1)}
    RESULTS[VL_KEY] = per_image
    print(f"  avg latency {RESOURCE[VL_KEY]['avg_latency_s']:.2f}s | peak GPU {g_run.peak:.0f} MB")
    free_gpu(predictor)
else:
    assert VL_REF_DIR and os.path.isdir(VL_REF_DIR), "RUN_VL=False needs a valid VL_REF_DIR with VL *.md files."
    md_files = glob.glob(os.path.join(VL_REF_DIR, "**", "*.md"), recursive=True)
    stem2img = {os.path.splitext(os.path.basename(p))[0]: os.path.basename(p) for p in images}
    for f in md_files:
        stem = os.path.splitext(os.path.basename(f))[0]
        img = stem2img.get(stem) or stem2img.get(stem.replace("_res", ""))
        if not img: continue
        txt = open(f, encoding="utf-8").read()
        VL_TEXT[img] = normtext(txt); VL_META[img] = {"n_tables": txt.lower().count("<table")}
    print(f"Loaded {len(VL_TEXT)} precomputed VL references from {VL_REF_DIR}")
assert VL_TEXT, "No VL reference text available."
print(f"VL reference ready for {len(VL_TEXT)} images.")

=== PaddleOCR-VL 1.6 (reference) ===
  loaded in 240.5s | model disk 1967 MB | GPU after load 12195 MB
  [100/100] report_20.jpg               
  avg latency 23.07s | peak GPU 14351 MB
VL reference ready for 100 images.


## Metrics
Quality is measured against VL 1.6 (pseudo-reference): word-level **Precision/Recall/F1** and character similarity on normalized text, plus **table-capture recall** for format. We also report GT-free proxies (regions, chars, confidence) and cross-model agreement. All tables are saved as CSV in the ZIP.

In [8]:
# ============================ COMPUTE METRICS ============================
import pandas as pd, numpy as np, os, json

CAND_KEYS = [m["key"] for m in MODELS]
NAME = {m["key"]: m["name"] for m in MODELS}; NAME[VL_KEY] = "PaddleOCR-VL-1.6"

# ---- resource / speed table (all models incl. VL) ----
res_df = pd.DataFrame([RESOURCE[k] for k in CAND_KEYS + ([VL_KEY] if VL_KEY in RESOURCE else [])])
res_df.to_csv(os.path.join(OUTPUT_ROOT, "resource_summary.csv"), index=False)

# ---- per-image long table (candidates) ----
rows = []
for k in CAND_KEYS:
    for b, v in RESULTS[k].items():
        rows.append({"model": NAME[k], "key": k, "image": b, "category": v["category"],
                     "latency": v["latency"], "regions": v["regions"], "chars": v["chars"],
                     "mean_conf": v["mean_conf"], "n_tables": v["n_tables"]})
long_df = pd.DataFrame(rows)
long_df.to_csv(os.path.join(OUTPUT_ROOT, "per_image_metrics.csv"), index=False)

# ---- quality vs VL 1.6 ----
qrows = []
for k in CAND_KEYS:
    for b, v in RESULTS[k].items():
        if b not in VL_TEXT: continue
        ref = VL_TEXT[b]; pred = v["norm_text"]
        P, R, F = word_prf(pred, ref)
        cs = char_sim(pred, ref)
        vl_tab = VL_META[b]["n_tables"]
        tab_recall = (min(v["n_tables"], vl_tab) / vl_tab) if vl_tab > 0 else np.nan
        qrows.append({"model": NAME[k], "key": k, "image": b, "category": v["category"],
                      "word_P": P, "word_R": R, "word_F1": F, "char_sim": cs,
                      "vl_tables": vl_tab, "model_tables": v["n_tables"], "table_recall": tab_recall})
qual_df = pd.DataFrame(qrows)
qual_df.to_csv(os.path.join(OUTPUT_ROOT, "quality_vs_vl_per_image.csv"), index=False)

qual_agg = (qual_df.groupby("model")
            .agg(word_P=("word_P", "mean"), word_R=("word_R", "mean"), word_F1=("word_F1", "mean"),
                 char_sim=("char_sim", "mean"), table_recall=("table_recall", "mean"))
            .reindex([NAME[k] for k in CAND_KEYS]).round(4))
qual_agg.to_csv(os.path.join(OUTPUT_ROOT, "quality_vs_vl_summary.csv"))

# ---- cross-model agreement (4x4 incl. VL) ----
AG_KEYS = CAND_KEYS + [VL_KEY]
def text_of(k, b):
    if k == VL_KEY: return VL_TEXT.get(b, "")
    return RESULTS[k].get(b, {}).get("norm_text", "")
common = [b for b in (RESULTS[CAND_KEYS[0]].keys()) if b in VL_TEXT]
agree = np.ones((len(AG_KEYS), len(AG_KEYS)))
for i, ki in enumerate(AG_KEYS):
    for j, kj in enumerate(AG_KEYS):
        if j <= i: continue
        s = np.mean([char_sim(text_of(ki, b), text_of(kj, b)) for b in common]) if common else np.nan
        agree[i, j] = agree[j, i] = s
agree_df = pd.DataFrame(agree, index=[NAME[k] for k in AG_KEYS], columns=[NAME[k] for k in AG_KEYS]).round(3)
agree_df.to_csv(os.path.join(OUTPUT_ROOT, "cross_model_agreement.csv"))

# ---- combined per-model summary (speed + coverage + quality) ----
cov = long_df.groupby("model").agg(avg_regions=("regions", "mean"), avg_chars=("chars", "mean"),
                                   mean_conf=("mean_conf", "mean"), avg_latency=("latency", "mean")).round(3)
combined = cov.join(qual_agg, how="left")
combined.to_csv(os.path.join(OUTPUT_ROOT, "combined_summary.csv"))

print("=== Resource / speed (per image averages) ===");            display(res_df.set_index("model").T)
print("\n=== Quality vs VL 1.6 (higher = closer to VL) ===");      display(qual_agg)
print("\n=== Cross-model agreement (char similarity) ===");        display(agree_df)
print("\n=== Combined per-model summary ===");                     display(combined)

=== Resource / speed (per image averages) ===


model,PP-OCRv6_small,PP-OCRv6_medium,PP-StructureV3 (OCRv5-server),PaddleOCR-VL-1.6
num_images,100.000,100.00,100.00,100.000
avg_latency_s,0.456,0.78,2.03,23.066
total_infer_s,45.600,78.00,203.00,2306.600
gpu_peak_MB,2139.200,3761.20,4485.20,14351.200
gpu_after_load_MB,595.200,987.20,2109.20,NaN
model_disk_MB,30.000,132.70,1024.20,1966.900
output_disk_MB,30.900,30.80,258.30,31.700
load_time_s,31.600,33.40,297.30,240.500



=== Quality vs VL 1.6 (higher = closer to VL) ===


,word_P,word_R,word_F1,char_sim,table_recall
model,,,,,
PP-OCRv6_small,0.8175,0.9040,0.8397,0.6552,0.0000
PP-OCRv6_medium,0.8251,0.9154,0.8493,0.6647,0.0000
PP-StructureV3 (OCRv5-server),0.7352,0.5812,0.6167,0.5209,0.7449



=== Cross-model agreement (char similarity) ===


,PP-OCRv6_small,PP-OCRv6_medium,PP-StructureV3 (OCRv5-server),PaddleOCR-VL-1.6
PP-OCRv6_small,1.000,0.909,0.469,0.655
PP-OCRv6_medium,0.909,1.000,0.479,0.665
PP-StructureV3 (OCRv5-server),0.469,0.479,1.000,0.521
PaddleOCR-VL-1.6,0.655,0.665,0.521,1.000



=== Combined per-model summary ===


,avg_regions,avg_chars,mean_conf,avg_latency,word_P,word_R,word_F1,char_sim,table_recall
model,,,,,,,,,
PP-OCRv6_medium,66.63,909.31,0.989,0.780,0.8251,0.9154,0.8493,0.6647,0.0000
PP-OCRv6_small,65.81,899.19,0.984,0.456,0.8175,0.9040,0.8397,0.6552,0.0000
PP-StructureV3 (OCRv5-server),67.11,675.75,0.952,2.030,0.7352,0.5812,0.6167,0.5209,0.7449


## Figures

In [ ]:
# ============================ FIGURES ============================
import matplotlib.pyplot as plt, numpy as np, os
FIG_DIR = os.path.join(OUTPUT_ROOT, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
cand_names = [NAME[k] for k in CAND_KEYS]
COL = {"PP-OCRv6_small": "#4C9BE8", "PP-OCRv6_medium": "#2E6FB0",
       "PP-StructureV3 (OCRv5-server)": "#E8894C", "PP-StructureV3 (OCRv6-medium)": "#C25E1E",
       "PaddleOCR-VL-1.6": "#7A7A7A"}
def cols(names): return [COL.get(n, "#888") for n in names]

def savefig(fig, fname):
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, fname), bbox_inches="tight"); plt.show()

# --- Fig 1: speed + GPU + disk (all models incl. VL) ---
alln = list(res_df["model"])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(alln, res_df["avg_latency_s"], color=cols(alln)); ax[0].set_title("Avg latency / image (s)"); ax[0].set_yscale("log")
ax[1].bar(alln, res_df["gpu_peak_MB"], color=cols(alln)); ax[1].set_title("GPU peak (MB)")
ax[2].bar(alln, res_df["model_disk_MB"], color=cols(alln)); ax[2].set_title("Model on disk (MB)")
for a in ax: a.tick_params(axis="x", rotation=30)
fig.suptitle("Resource & speed", fontweight="bold"); savefig(fig, "fig1_resources.png")

# --- Fig 2: quality vs VL (F1 / char-sim / table recall) ---
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(cand_names, qual_agg["word_F1"], color=cols(cand_names)); ax[0].set_title("Word F1 vs VL"); ax[0].set_ylim(0, 1)
ax[1].bar(cand_names, qual_agg["char_sim"], color=cols(cand_names)); ax[1].set_title("Char similarity vs VL"); ax[1].set_ylim(0, 1)
ax[2].bar(cand_names, qual_agg["table_recall"], color=cols(cand_names)); ax[2].set_title("Table-capture recall vs VL"); ax[2].set_ylim(0, 1)
for a in ax: a.tick_params(axis="x", rotation=30)
fig.suptitle("Quality vs PaddleOCR-VL 1.6 (higher = closer to VL)", fontweight="bold"); savefig(fig, "fig2_quality_vs_vl.png")

# --- Fig 3: quality vs cost scatter (the trade-off chart) ---
fig, ax = plt.subplots(figsize=(7, 5))
for k in CAND_KEYS:
    n = NAME[k]
    x = RESOURCE[k]["avg_latency_s"]; y = float(qual_agg.loc[n, "word_F1"])
    ax.scatter(x, y, s=160, color=COL.get(n), zorder=3); ax.annotate(n, (x, y), xytext=(6, 6), textcoords="offset points")
ax.set_xlabel("avg latency / image (s)  → cheaper is left"); ax.set_ylabel("Word F1 vs VL 1.6  → better is up")
ax.set_title("Quality-vs-cost: closest to VL 1.6 per second", fontweight="bold"); ax.grid(alpha=0.3)
savefig(fig, "fig3_quality_vs_cost.png")

# --- Fig 4: latency by category (grouped) ---
piv = long_df.pivot_table(index="category", columns="model", values="latency", aggfunc="mean")
piv = piv[[n for n in cand_names if n in piv.columns]]
fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(piv.index)); w = 0.8 / len(piv.columns)
for i, n in enumerate(piv.columns):
    ax.bar(x + i*w, piv[n], w, label=n, color=COL.get(n))
ax.set_xticks(x + w*(len(piv.columns)-1)/2); ax.set_xticklabels(piv.index, rotation=20)
ax.set_ylabel("avg latency (s)"); ax.set_title("Latency by document category", fontweight="bold"); ax.legend()
savefig(fig, "fig4_latency_by_category.png")

# --- Fig 5: recognition confidence distribution (models that report it) ---
conf_keys = [k for k in CAND_KEYS if long_df[long_df.key == k]["mean_conf"].notna().any()]
fig, ax = plt.subplots(figsize=(8, 4.5))
data = [long_df[long_df.key == k]["mean_conf"].dropna() for k in conf_keys]
bp = ax.boxplot(data, patch_artist=True)
ax.set_xticks(range(1, len(conf_keys)+1)); ax.set_xticklabels([NAME[k] for k in conf_keys], rotation=20)
for patch, k in zip(bp["boxes"], conf_keys): patch.set_facecolor(COL.get(NAME[k]))
ax.set_ylabel("per-image mean confidence"); ax.set_title("Recognition confidence distribution", fontweight="bold")
savefig(fig, "fig5_confidence.png")

# --- Fig 6: cross-model agreement heatmap ---
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(agree_df.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(agree_df))); ax.set_xticklabels(agree_df.columns, rotation=40, ha="right")
ax.set_yticks(range(len(agree_df))); ax.set_yticklabels(agree_df.index)
for i in range(len(agree_df)):
    for j in range(len(agree_df)):
        ax.text(j, i, f"{agree_df.values[i,j]:.2f}", ha="center", va="center",
                color="white" if agree_df.values[i,j] < 0.6 else "black", fontsize=9)
fig.colorbar(im, fraction=0.046); ax.set_title("Cross-model text agreement", fontweight="bold")
savefig(fig, "fig6_agreement.png")

# --- Fig 7: coverage (regions + chars) ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(cand_names, [long_df[long_df.model==n]["regions"].mean() for n in cand_names], color=cols(cand_names))
ax[0].set_title("Avg text regions / image")
ax[1].bar(cand_names, [long_df[long_df.model==n]["chars"].mean() for n in cand_names], color=cols(cand_names))
ax[1].set_title("Avg chars extracted / image")
for a in ax: a.tick_params(axis="x", rotation=30)
fig.suptitle("Text coverage", fontweight="bold"); savefig(fig, "fig7_coverage.png")
print("Figures saved to", FIG_DIR)

## Sample outputs (5 images)
Original vs each model's annotated overlay, so the numbers above can be sanity-checked by eye.

In [ ]:
# ============================ SAMPLE VISUALS ============================
import matplotlib.pyplot as plt, matplotlib.image as mpimg, glob, os

def find_overlay(model_key, stem):
    d = os.path.join(OUTPUT_ROOT, model_key)
    cands = glob.glob(os.path.join(d, "**", f"{stem}*.jpg"), recursive=True) + \
            glob.glob(os.path.join(d, "**", f"{stem}*.png"), recursive=True)
    # prefer an OCR/overlay style image
    pref = [c for c in cands if any(t in os.path.basename(c).lower() for t in ("ocr", "overlay", "res"))]
    return (pref or cands or [None])[0]

panel_keys = CAND_KEYS + ([VL_KEY] if VL_KEY in RESULTS else [])
for p in SAMPLES:
    stem = os.path.splitext(os.path.basename(p))[0]
    imgs = [("original", p)] + [(NAME[k], find_overlay(k, stem)) for k in panel_keys]
    imgs = [(t, f) for t, f in imgs if f and os.path.exists(f)]
    n = len(imgs); fig, ax = plt.subplots(1, n, figsize=(4.2*n, 5))
    if n == 1: ax = [ax]
    for a, (title, f) in zip(ax, imgs):
        a.imshow(mpimg.imread(f)); a.axis("off"); a.set_title(title, fontsize=9)
    fig.suptitle(os.path.basename(p), fontsize=10, fontweight="bold"); plt.tight_layout(); plt.show()

## Text / layout sample: StructureV3 vs VL 1.6
Extracted Markdown for one sample (tables shown as HTML). A full `<stem>.html` page — text + reconstructed tables in reading order — is saved per image for both StructureV3 and VL.

In [ ]:
# ============================ TEXT / MARKDOWN SNIPPET ============================
import os, glob
def read_md(model_key, stem):
    fs = glob.glob(os.path.join(OUTPUT_ROOT, model_key, "**", f"{stem}*.md"), recursive=True)
    return open(fs[0], encoding="utf-8").read() if fs else "(no markdown produced by this model)"

sample = SAMPLES[0]; stem = os.path.splitext(os.path.basename(sample))[0]
struct_keys = [m["key"] for m in MODELS if m["kind"] == "struct"]
print(f"### {os.path.basename(sample)}   (full HTML pages saved as <stem>.html per model)\n")
for k in (struct_keys + ([VL_KEY] if VL_KEY in RESULTS else [])):
    print("="*70); print(NAME[k]); print("="*70)
    print(read_md(k, stem)[:1400], "\n")
# candidate flat-text vs VL text closeness numbers for this image
b = os.path.basename(sample)
if b in VL_TEXT:
    print("-"*70, "\nWord-F1 / char-sim vs VL 1.6 for this image:")
    for k in CAND_KEYS:
        pred = RESULTS[k][b]["norm_text"]; P,R,F = word_prf(pred, VL_TEXT[b])
        print(f"  {NAME[k]:<18} F1={F:.3f}  char_sim={char_sim(pred, VL_TEXT[b]):.3f}")

## Verdict

In [ ]:
# ============================ AUTO VERDICT ============================
import numpy as np
best_f1  = qual_agg["word_F1"].idxmax()
best_tab = qual_agg["table_recall"].idxmax()
fastest  = min(CAND_KEYS, key=lambda k: RESOURCE[k]["avg_latency_s"])
# efficiency = F1 per second (closeness to VL per unit cost)
eff = {NAME[k]: float(qual_agg.loc[NAME[k], "word_F1"]) / max(RESOURCE[k]["avg_latency_s"], 1e-6) for k in CAND_KEYS}
best_eff = max(eff, key=eff.get)

print("VERDICT — which light model best stands in for VL 1.6 on this task")
print("="*66)
print(f"Closest to VL (word F1)      : {best_f1}  ({qual_agg.loc[best_f1,'word_F1']:.3f})")
print(f"Best format capture (tables) : {best_tab}  ({qual_agg.loc[best_tab,'table_recall']:.3f})")
print(f"Fastest                      : {NAME[fastest]}  ({RESOURCE[fastest]['avg_latency_s']:.2f} s/img)")
print(f"Best F1-per-second           : {best_eff}  ({eff[best_eff]:.3f})")
print()
print("Reading it: OCRv6 (small/medium) capture text only — expect ~0 table recall, so they")
print("approximate VL's *text* but not its *format*. StructureV3 is the only candidate that")
print("reproduces VL's tables/Markdown. Pick by whether your task needs format or just text,")
print("then trade the F1-vs-latency scatter (fig 3) for your speed/GPU budget.")

In [ ]:
# ============================ ZIP EVERYTHING ============================
import shutil, os
from IPython.display import FileLink
zip_file = shutil.make_archive(ZIP_PATH, "zip", OUTPUT_ROOT)
print(f"Wrote {zip_file}  ({os.path.getsize(zip_file)/1024**2:.1f} MB)")
print("Contains: per-model outputs (json/img/markdown + HTML pages for StructureV3 & VL),")
print("          metric CSVs, figures/PNGs, logs/ (captured Paddle chatter), all models' results.")
FileLink(zip_file)